# **Kaggle – DataTops®**
Tu TA ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Aspectos importantes
- Última submission:
    - Mañana: 17 de febrero a las 5pm
    - Tarde: 19 de febrero a las 5pm
- **Enlace de la competición**: https://www.kaggle.com/t/c5cc87b50c4b4770bdc8f5acbe15577d
- **Requisito**: Estar registrado en [Kaggle](https://www.kaggle.com/)

## Métrica:
El error cuadrático medio (RMSE, por sus siglas en inglés) es una medida de la desviación estándar de los residuos (errores de predicción). Los residuos representan la diferencia entre los valores observados y los valores predichos por el modelo. El RMSE indica qué tan dispersos están estos errores: cuanto menor es el RMSE, más cercanas están las predicciones a los valores reales. En otras palabras, el RMSE mide qué tan bien se ajusta la línea de regresión a los datos.


$$ RMSE = \sqrt{\frac{1}{n}\Sigma_{i=1}^{n}{\Big(\frac{d_i -f_i}{\sigma_i}\Big)^2}}$$


## 1. Librerías

In [98]:
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

import urllib.request
import matplotlib.pyplot as plt
import MyFunctions2 as mf    
import category_encoders as ce

## 2. Datos

In [99]:
# Para que funcione necesitas bajarte los archivos de datos de Kaggle
df = pd.read_csv("./data/train.csv")
X_pred = pd.read_csv("./data/test.csv")

### 2.1 Exploración de los datos

In [100]:
# Hacemos que nuestro índice sea el id
df.set_index("laptop_ID", inplace=True)
X_pred.set_index("laptop_ID", inplace=True)

y_test_pred = pd.read_csv("./data/test2.csv", sep=';')
y_test_pred.set_index("laptop_ID", inplace=True)    



In [101]:
df["Price_in_euros"] = np.log1p(df["Price_in_euros"])
#y_test_pred["Price_in_euros"] = np.log1p(y_test_pred["Price_in_euros"])

### 2.3 Dividir X_train, X_test, y_train, y_test

In [102]:
train_set, test_set = train_test_split(df, test_size = 0.2, random_state = 42)

In [103]:
target_col = "Price_in_euros"
features = [c for c in train_set.columns if c != target_col]

y_train = train_set[target_col]
y_test = test_set[target_col]

X_train = train_set[features]
X_test = test_set[features]


In [104]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 729 entries, 1118 to 418
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           729 non-null    object 
 1   Product           729 non-null    object 
 2   TypeName          729 non-null    object 
 3   Inches            729 non-null    float64
 4   ScreenResolution  729 non-null    object 
 5   Cpu               729 non-null    object 
 6   Ram               729 non-null    object 
 7   Memory            729 non-null    object 
 8   Gpu               729 non-null    object 
 9   OpSys             729 non-null    object 
 10  Weight            729 non-null    object 
dtypes: float64(1), object(10)
memory usage: 68.3+ KB


In [105]:
def transformar_datos(df):

    mf.screen_resolution_split(df)
    mf.replace_value(df, "Ram", "GB", "", int)

    mf.replace_value(df, "Weight", "kg", "", float)
    mf.memory_split(df)

    mf.procesar_cpu_caotico(df, "Cpu")

    # df['Cpu_Tier'] = df['cpu_familia'].apply(mf.categorizar_procesador)
    # df.drop(["cpu_familia"], axis=1, inplace=True)
    
    mf.gpu_split3(df)
    df.drop(["Product"], axis=1, inplace=True)

    # df['Performance_Weight_Ratio'] = df['cpu_ghz'] / df['Weight_kg']
    # df['Cpu_Performance_Index'] = df['Cpu_Tier'] * df['cpu_ghz']
    # df['Ram_GHz_Ratio'] = df['Ram_GB'] / df['cpu_ghz']
    # df['Power_Density'] = (df['Ram_GB'] * df['cpu_ghz']) / df['Weight_kg']

    # df.drop(["cpu_ghz"], axis=1, inplace=True)
    # df.drop(["Weight_kg"], axis=1, inplace=True)

    

    return df


In [108]:
from lightgbm import LGBMRegressor
from sklearn.calibration import LabelEncoder
from sklearn.metrics import mean_squared_error


def transformar_datos2(df):

    df["Touchscreen"] = df["ScreenResolution"].str.contains("Touchscreen").astype(int)
    df["IPS"] = df["ScreenResolution"].str.contains("IPS").astype(int)

    res = df["ScreenResolution"].str.extract(r'(\d+)x(\d+)')
    df["ResX"] = res[0].astype(float)
    df["ResY"] = res[1].astype(float)

    df["PPI"] = ((df["ResX"]**2 + df["ResY"]**2)**0.5) / df["Inches"]

    df["Cpu_brand"] = df["Cpu"].str.split().str[0]
    df["Cpu_family"] = df["Cpu"].str.extract(r'(i3|i5|i7|i9|Ryzen|Pentium|Celeron|Atom)')
    df["Cpu_speed"] = df["Cpu"].str.extract(r'(\d\.\d+)GHz').astype(float)

    df["Gpu_brand"] = df["Gpu"].str.split().str[0]
    df["Gpu_dedicated"] = df["Gpu_brand"].apply(lambda x: 0 if x=="Intel" else 1)

    def parse_memory(mem):
        mem = mem.replace("GB", "").replace("TB", "000")
        parts = mem.split("+")
        hdd = ssd = flash = hybrid = 0
        for p in parts:
            p = p.strip()
            if "HDD" in p:
                hdd += float(p.split()[0])
            elif "SSD" in p:
                ssd += float(p.split()[0])
            elif "Flash" in p:
                flash += float(p.split()[0])
            elif "Hybrid" in p:
                hybrid += float(p.split()[0])
        return pd.Series([hdd, ssd, flash, hybrid])

    df[["HDD", "SSD", "Flash", "Hybrid"]] = df["Memory"].apply(parse_memory)
    df["Total_Storage"] = df[["HDD","SSD","Flash","Hybrid"]].sum(axis=1)
  
    mf.replace_value(df, "Ram", "GB", "", int)

    mf.replace_value(df, "Weight", "kg", "", float)

    df["Cpu_gen"] = df["Cpu"].str.extract(r'(\d{4})') 
    df["Cpu_gen"] = df["Cpu_gen"].astype(float) // 100 
    df["Cpu_gen"] = df["Cpu_gen"].fillna(df["Cpu_gen"].median())

    df["Gpu_model_num"] = df["Gpu"].str.extract(r'(\d{3,4})').astype(float) 
    df["Gpu_model_num"] = df["Gpu_model_num"].fillna(0)
    df["Gpu_is_integrated"] = (df["Gpu_brand"] == "Intel").astype(int)
    df["Cpu_power"] = df["Cpu_gen"] * df["Cpu_speed"]
    df["Gpu_power"] = df["Gpu_dedicated"] * df["Gpu_model_num"]

    df["Has_SSD"] = (df["SSD"] > 0).astype(int) 
    df["Has_HDD"] = (df["HDD"] > 0).astype(int)
    df["Storage_speed"] = df["Has_SSD"] * 2 + df["Has_HDD"] * 1

    df["Weight_per_inch"] = df["Weight_kg"] / df["Inches"]
    df["Ram_per_kg"] = df["Ram_GB"] / df["Weight_kg"]

    def resolution_category(x): 
        if "3840" in x: return "4K" 
        if "3200" in x or "2560" in x: return "2K" 
        if "1920" in x: return "FHD" 
        return "HD" 
    
    #df["ResCategory"] = df["ScreenResolution"].apply(resolution_category)
    df["Weight_norm"] = df["Weight_kg"] / df["Weight_kg"].max() 
    df["Ram_norm"] = df["Ram_GB"] / 32

    df.drop(["Product","ScreenResolution", "Cpu", "Gpu", "Memory"], axis=1, inplace=True)

    return df

In [109]:
from sklearn.preprocessing import RobustScaler

X_train = transformar_datos2(X_train)
X_test = transformar_datos2(X_test)
X_pred = transformar_datos2(X_pred)

X_train.info()


<class 'pandas.core.frame.DataFrame'>
Index: 729 entries, 1118 to 418
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Company            729 non-null    object 
 1   TypeName           729 non-null    object 
 2   Inches             729 non-null    float64
 3   OpSys              729 non-null    object 
 4   Touchscreen        729 non-null    int64  
 5   IPS                729 non-null    int64  
 6   ResX               729 non-null    float64
 7   ResY               729 non-null    float64
 8   PPI                729 non-null    float64
 9   Cpu_brand          729 non-null    object 
 10  Cpu_family         688 non-null    object 
 11  Cpu_speed          686 non-null    float64
 12  Gpu_brand          729 non-null    object 
 13  Gpu_dedicated      729 non-null    int64  
 14  HDD                729 non-null    float64
 15  SSD                729 non-null    float64
 16  Flash              729 non-n

In [110]:
X_train.head()

,Company,TypeName,Inches,OpSys,Touchscreen,IPS,ResX,ResY,PPI,Cpu_brand,...,Gpu_is_integrated,Cpu_power,Gpu_power,Has_SSD,Has_HDD,Storage_speed,Weight_per_inch,Ram_per_kg,Weight_norm,Ram_norm
laptop_ID,,,,,,,,,,,,,,,,,,,,,
1118,HP,Workstation,17.3,Windows 7,0,1,1920.0,1080.0,127.335675,Intel,...,0,174.2,6150.0,0,1,1,0.173410,2.666667,0.638298,0.250
153,Dell,Gaming,15.6,Windows 10,0,0,1920.0,1080.0,141.211998,Intel,...,0,215.6,1050.0,1,0,2,0.164103,6.250000,0.544681,0.500
275,Apple,Ultrabook,13.3,macOS,0,1,2560.0,1600.0,226.983005,Intel,...,1,208.8,0.0,1,0,2,0.103008,5.839416,0.291489,0.250
1100,HP,Notebook,14.0,Windows 7,0,0,1920.0,1080.0,157.350512,Intel,...,1,142.6,0.0,0,1,1,0.110000,2.597403,0.327660,0.125
131,Dell,Notebook,17.3,Windows 10,0,0,1920.0,1080.0,127.335675,Intel,...,0,153.0,530.0,1,1,3,0.161850,5.714286,0.595745,0.500


In [111]:
# cat_cols = X_train.select_dtypes(include='object').columns.tolist()

cat_cols = ["Company","TypeName","OpSys","Cpu_brand","Cpu_family","Gpu_brand"]
encoder = ce.TargetEncoder(cols=cat_cols)
X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols], y_train)
X_test[cat_cols] = encoder.transform(X_test[cat_cols])
X_pred[cat_cols] = encoder.transform(X_pred[cat_cols])




In [112]:
for col in X_train.select_dtypes(include='object').columns:
    print(f"{col}: {X_train[col].nunique()} categorías")

In [113]:
# Eliminar variables con baja correlación respecto al target
# Solo considerar columnas numéricas para la correlación
def drop_low_correlation_features(X, y, threshold=0.05):
    X_numeric = X.select_dtypes(include=[np.number])
    corrs = X_numeric.corrwith(y).abs()
    low_corr_cols = corrs[corrs < threshold].index.tolist()
    print(f"Eliminando {len(low_corr_cols)} columnas con correlación < {threshold}: {low_corr_cols}")
    return X.drop(columns=low_corr_cols), low_corr_cols

X_train, dropped_low_corr = drop_low_correlation_features(X_train, y_train, threshold=0.05)
X_test = X_test.drop(columns=dropped_low_corr)
X_pred = X_pred.drop(columns=dropped_low_corr)


Eliminando 3 columnas con correlación < 0.05: ['Inches', 'Flash', 'Hybrid']


In [114]:
# Calcular la correlación de cada feature con el target
correlation = X_train.corrwith(y_train)
correlation.sort_values(ascending=False, inplace=True)
correlation

Cpu_family           0.768871
Ram_GB               0.677051
Ram_norm             0.677051
TypeName             0.641322
SSD                  0.633920
Ram_per_kg           0.629793
Storage_speed        0.605618
Has_SSD              0.592999
ResY                 0.559496
ResX                 0.557786
Cpu_power            0.539200
Cpu_speed            0.500856
PPI                  0.490018
Cpu_gen              0.452715
Gpu_brand            0.355235
OpSys                0.349454
Company              0.346207
Gpu_power            0.330327
IPS                  0.292776
Gpu_model_num        0.284557
Touchscreen          0.234763
Cpu_brand            0.223724
Gpu_dedicated        0.195111
Total_Storage        0.181288
Weight_norm          0.140551
Weight_kg            0.140551
Weight_per_inch      0.132693
HDD                 -0.063261
Has_HDD             -0.160063
Gpu_is_integrated   -0.195111
dtype: float64

In [115]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Entrenar el modelo Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

def predecir(model, X_test, y_test):
    # Predecir sobre el test
    y_pred = model.predict(X_test)

    y_pred = np.expm1(y_pred)
    # Calcular RMSE en escala logarítmica
    rmse_log = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"RMSE (log1p): {rmse_log:.4f}")

    print(np.sqrt(mean_squared_error(y_test, y_pred))) 
    print(np.sqrt(mean_squared_error(y_test, np.mean(y_test)*np.ones_like(y_test))))

    # # Calcular RMSE en escala original (deshacer log1p)
    # y_test_exp = np.expm1(y_test)
    # y_pred_exp = np.expm1(y_pred)
    # rmse_original = np.sqrt(mean_squared_error(y_test_exp, y_pred_exp))
    # print(f"RMSE (escala original): {rmse_original:.2f}")

In [116]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error
import numpy as np

# Búsqueda de hiperparámetros ampliada para RandomForest
param_grid = {
    'n_estimators': [100, 200, 400],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False],
    'random_state': [42]

}

rf = RandomForestRegressor(random_state=42)
grid_search_rf = GridSearchCV(
    rf, param_grid, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=2
)
grid_search_rf.fit(X_train, y_train)


Fitting 3 folds for each of 1152 candidates, totalling 3456 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'bootstrap': [True, False], 'max_depth': [None, 10, ...], 'max_features': ['sqrt', 'log2', ...], 'min_samples_leaf': [1, 2, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computa

In [117]:
import optuna
import xgboost as xgb
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error

# Optuna para XGBoost

def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 10),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'random_state': 42
    }
    model = xgb.XGBRegressor(**params)
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='neg_root_mean_squared_error').mean()
    return score

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=75)

print("Mejores hiperparámetros XGBoost:", study.best_params)

# Entrenar modelo final con los mejores hiperparámetros
best_params = study.best_params
model_xgb_optuna = xgb.XGBRegressor(**best_params)
model_xgb_optuna.fit(X_train, y_train)




[I 2026-02-16 23:53:09,429] A new study created in memory with name: no-name-1466e7d3-132d-4ec9-a9ce-714e5ab95c25
[I 2026-02-16 23:53:09,648] Trial 0 finished with value: -0.2643433634321725 and parameters: {'max_depth': 7, 'learning_rate': 0.24517932047070642, 'n_estimators': 466, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'gamma': 1.5599452033620265, 'reg_alpha': 0.5808361216819946, 'reg_lambda': 8.661761457749352, 'min_child_weight': 7}. Best is trial 0 with value: -0.2643433634321725.
[I 2026-02-16 23:53:10,029] Trial 1 finished with value: -0.28297259501253563 and parameters: {'max_depth': 12, 'learning_rate': 0.005439667429522981, 'n_estimators': 585, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381, 'gamma': 1.8182496720710062, 'reg_alpha': 1.8340450985343382, 'reg_lambda': 3.0424224295953772, 'min_child_weight': 6}. Best is trial 0 with value: -0.2643433634321725.
[I 2026-02-16 23:53:10,250] Trial 2 finished with value: -0.33

Mejores hiperparámetros XGBoost: {'max_depth': 9, 'learning_rate': 0.20487125223768562, 'n_estimators': 191, 'subsample': 0.8767299351668862, 'colsample_bytree': 0.9996503871976347, 'gamma': 0.0015156320097993586, 'reg_alpha': 0.6989984087130298, 'reg_lambda': 6.526380416402371, 'min_child_weight': 10}


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.9996503871976347
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import

In [118]:
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

def objective_lgb(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 100, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 30),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
        'random_state': 42,
        'verbose': -1
    }
    model = lgb.LGBMRegressor(**params)
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='neg_root_mean_squared_error').mean()
    return score

study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(objective_lgb, n_trials=50)

print("Mejores hiperparámetros LightGBM:", study_lgb.best_params)

best_params_lgb = study_lgb.best_params
model_lgb_optuna = lgb.LGBMRegressor(**best_params_lgb)
model_lgb_optuna.fit(X_train, y_train)


[I 2026-02-16 23:53:37,406] A new study created in memory with name: no-name-9806acf6-f08f-4184-ad30-d078f0d89d84
C:\Users\jvend\AppData\Local\Temp\ipykernel_34596\848519506.py:7: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'num_leaves': trial.suggest_int('num_leaves', 20, 100, 150),
c:\Users\jvend\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\distributions.py:684: UserWarning: The distribution is specified by [20, 100] and step=150, but the range is not divisible by `step`. It will be replaced with [20, 20].
  optuna_warn(
[I 2026-02-16 23:53:37,504] Trial 0 finished with value: 

Mejores hiperparámetros LightGBM: {'num_leaves': 20, 'max_depth': 7, 'learning_rate': 0.07120409347777255, 'n_estimators': 255, 'min_child_samples': 11, 'subsample': 0.8891490606170778, 'colsample_bytree': 0.7583648488293474, 'reg_alpha': 0.07493016881369954, 'reg_lambda': 3.841235401211049}


,boosting_type,'gbdt'
,num_leaves,20
,max_depth,7
,learning_rate,0.07120409347777255
,n_estimators,255
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,11


In [119]:
y_test_pred = pd.read_csv("./data/test2.csv", sep=';')
y_test_pred.set_index("laptop_ID", inplace=True)    

#predecir(grid_search_rf, X_pred, y_test_pred)
# Evaluación en test
predecir(model_xgb_optuna, X_pred, y_test_pred)
predecir(model_lgb_optuna, X_pred, y_test_pred)

RMSE (log1p): 299.8463
299.8463148014329
723.3320331742951
RMSE (log1p): 318.1985
318.1984797903868
723.3320331742951


In [120]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error

def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 40),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'random_state': 42
    }
    model = RandomForestRegressor(**params)
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='neg_root_mean_squared_error').mean()
    return score

study_rf = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(objective_rf, n_trials=50)

print("Mejores hiperparámetros RandomForest (Optuna):", study_rf.best_params)

best_params_rf = study_rf.best_params
model_rf_optuna = RandomForestRegressor(**best_params_rf)
model_rf_optuna.fit(X_train, y_train)


[I 2026-02-16 23:53:55,464] A new study created in memory with name: no-name-16df1d3a-6576-4546-91d3-194c1392db5c
[I 2026-02-16 23:53:56,099] Trial 0 finished with value: -0.2441116180504602 and parameters: {'n_estimators': 287, 'max_depth': 39, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 0 with value: -0.2441116180504602.
[I 2026-02-16 23:53:56,804] Trial 1 finished with value: -0.2514732722657953 and parameters: {'n_estimators': 454, 'max_depth': 5, 'min_samples_split': 20, 'min_samples_leaf': 9, 'max_features': 'sqrt', 'bootstrap': False}. Best is trial 0 with value: -0.2441116180504602.
[I 2026-02-16 23:53:58,223] Trial 2 finished with value: -0.2320440892672674 and parameters: {'n_estimators': 316, 'max_depth': 15, 'min_samples_split': 13, 'min_samples_leaf': 2, 'max_features': None, 'bootstrap': True}. Best is trial 2 with value: -0.2320440892672674.
[I 2026-02-16 23:53:59,649] Trial 3 finished with value: -0.245367006

Mejores hiperparámetros RandomForest (Optuna): {'n_estimators': 544, 'max_depth': 38, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'bootstrap': False}


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",544
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",38
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",3
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsampl

In [121]:
predecir(model_rf_optuna, X_pred, y_test_pred)

RMSE (log1p): 319.8967
319.8966836476069
723.3320331742951


In [122]:
import optuna
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
import numpy as np

# Función objetivo para Optuna
def objective(trial):

    params = {
        "iterations": trial.suggest_int("iterations", 800, 2500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "random_strength": trial.suggest_float("random_strength", 0.5, 2.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count": trial.suggest_int("border_count", 64, 254),
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "verbose": False,
        "random_seed": 42
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmses = []

    for train_idx, valid_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

        model = CatBoostRegressor(**params)
        model.fit(X_tr, y_tr)

        # Predicción en log
        y_pred_log = model.predict(X_val)
        y_pred = np.expm1(y_pred_log)
        y_val = np.expm1(y_val)


        # RMSE real
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        rmses.append(rmse)

    return np.mean(rmses)

# Ejecutar Optuna
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=60)

print("Mejores parámetros:", study.best_params)
print("Mejor RMSE:", study.best_value)


[I 2026-02-16 23:54:56,549] A new study created in memory with name: no-name-5a8cef14-c975-4734-9966-0a629e7a94ac
[I 2026-02-16 23:54:59,795] Trial 0 finished with value: 284.2844802131211 and parameters: {'iterations': 1370, 'learning_rate': 0.014168492380332874, 'depth': 4, 'l2_leaf_reg': 4.875518504369687, 'random_strength': 1.5207695888103334, 'bagging_temperature': 0.2979818924340496, 'border_count': 111}. Best is trial 0 with value: 284.2844802131211.
[I 2026-02-16 23:55:05,663] Trial 1 finished with value: 277.5631300061019 and parameters: {'iterations': 1338, 'learning_rate': 0.04741141696005516, 'depth': 5, 'l2_leaf_reg': 9.019192653136676, 'random_strength': 1.2012871373940865, 'bagging_temperature': 0.18839182022543677, 'border_count': 250}. Best is trial 1 with value: 277.5631300061019.
[I 2026-02-16 23:55:09,213] Trial 2 finished with value: 287.1341544433808 and parameters: {'iterations': 859, 'learning_rate': 0.03463369977252167, 'depth': 5, 'l2_leaf_reg': 9.257329122502

Mejores parámetros: {'iterations': 1781, 'learning_rate': 0.048143364090063284, 'depth': 4, 'l2_leaf_reg': 1.4392964169574116, 'random_strength': 1.2182814833610052, 'bagging_temperature': 0.22284739590632852, 'border_count': 241}
Mejor RMSE: 258.8903479396476


In [123]:
best_params = study.best_params 
best_params["loss_function"] = "RMSE" 
best_params["eval_metric"] = "RMSE" 
best_params["verbose"] = False 
best_params["random_seed"] = 42 
model = CatBoostRegressor(**best_params) 
# Entrenar en todo el dataset 
model.fit(X_train, y_train)
predecir(model, X_pred, y_test_pred)

RMSE (log1p): 291.8891
291.88911094526196
723.3320331742951


In [130]:
y_pred = model.predict(X_pred)
y_pred = np.expm1(y_pred)

rmse_log = np.sqrt(mean_squared_error(y_test_pred, y_pred))
print(f"RMSE (log1p): {rmse_log:.4f}")

submission = pd.DataFrame(y_pred, columns=["Price_in_euros"], index=X_pred.index)
submission = submission.reset_index()

RMSE (log1p): 291.8891


In [127]:
def chequeador(df_to_submit):
    """
    Esta función se asegura de que tu submission tenga la forma requerida por Kaggle.

    Si es así, se guardará el dataframe en un `csv` y estará listo para subir a Kaggle.

    Si no, LEE EL MENSAJE Y HAZLE CASO.

    Si aún no:
    - apaga tu ordenador,
    - date una vuelta,
    - enciendelo otra vez,
    - abre este notebook y
    - leelo todo de nuevo.
    Todos nos merecemos una segunda oportunidad. También tú.
    """
    if df_to_submit.shape == sample.shape:
        if df_to_submit.columns.all() == sample.columns.all():
            if df_to_submit.laptop_ID.all() == sample.laptop_ID.all():
                print("You're ready to submit!")
                df_to_submit.to_csv("submission.csv", index = False) #muy importante el index = False
                urllib.request.urlretrieve("https://www.mihaileric.com/static/evaluation-meme-e0a350f278a36346e6d46b139b1d0da0-ed51e.jpg", "gfg.png")
                img = Image.open("gfg.png")
                img.show()
            else:
                print("Check the ids and try again")
        else:
            print("Check the names of the columns and try again")
    else:
        print("Check the number of rows and/or columns and try again")
        print("\nMensaje secreto del TA: No me puedo creer que después de todo este notebook hayas hecho algún cambio en las filas de `test.csv`. Lloro.")

In [129]:
sample = pd.read_csv("data/sample_submission.csv")
chequeador(submission)

You're ready to submit!


In [124]:
# Ensemble: promedio de XGBoost, RandomForest y LightGBM (Optuna)
# Asegúrate de tener entrenados: model_xgb_optuna, model_rf_optuna, model_lgb_optuna

pred_xgb = model_xgb_optuna.predict(X_pred)
pred_rf = model_rf_optuna.predict(X_pred)
pred_lgb = model_lgb_optuna.predict(X_pred)
pred_cat = model.predict(X_pred)


ensemble_pred_3 = ( pred_xgb + pred_rf + pred_lgb + pred_cat ) / 4
ensemble_pred_3 = np.expm1(ensemble_pred_3)
#ensemble_pred_3 = ( pred_xgb + pred_rf ) / 2

# y_test_exp = np.expm1(y_test_pred)
# y_pred_exp = np.expm1(ensemble_pred_3)
rmse_original = np.sqrt(mean_squared_error(y_test_pred, ensemble_pred_3))
print(f"RMSE Ensemble (4 modelos): {rmse_original:.2f}")

RMSE Ensemble (4 modelos): 300.43


In [73]:
X_train.head()

,Company,TypeName,OpSys,Touchscreen,IPS,ResX,ResY,PPI,Cpu_brand,Cpu_family,...,Total_Storage,Ram_GB,Weight_kg,Cpu_gen,Gpu_model_num,Has_SSD,Has_HDD,ResCategory,Weight_norm,Ram_norm
laptop_ID,,,,,,,,,,,,,,,,,,,,,
1118,6.841171,7.097950,7.091761,0,1,1920.0,1080.0,127.335675,6.855549,7.263482,...,1000.0,8,3.00,67.0,6150.0,0,1,6.996008,0.638298,0.250
153,6.899758,7.369785,6.886584,0,0,1920.0,1080.0,141.211998,6.855549,7.263482,...,512.0,16,2.56,77.0,1050.0,1,0,6.996008,0.544681,0.500
275,6.961628,7.282348,6.936435,0,1,2560.0,1600.0,226.983005,6.855549,6.864683,...,512.0,8,1.37,NaN,550.0,1,0,7.341643,0.291489,0.250
1100,6.841171,6.516254,7.091761,0,0,1920.0,1080.0,157.350512,6.855549,6.864683,...,500.0,4,1.54,62.0,520.0,0,1,6.996008,0.327660,0.125
131,6.899758,6.516254,6.886584,0,0,1920.0,1080.0,127.335675,6.855549,7.263482,...,2256.0,16,2.80,85.0,530.0,1,1,6.996008,0.595745,0.500


In [32]:
# graba xtrain, y_train, xtest, ytest, xpred en csv para usarlo luego
X_train.to_csv("./data/X_train_processed.csv", index=True)
y_train.to_csv("./data/y_train.csv", index=True)
X_test.to_csv("./data/X_test_processed.csv", index=True)
y_test.to_csv("./data/y_test.csv", index=True)
X_pred.to_csv("./data/X_pred_processed.csv", index=True)


In [33]:
ensemble_pred_3 = pd.Series(ensemble_pred_3, index=X_pred.index)
ensemble_pred_3.to_csv("./data/ensemble_pred_3.csv", index=True)

PermissionError: [Errno 13] Permission denied: './data/ensemble_pred_3.csv'

In [34]:
y_train.describe()

count    729.000000
mean       6.826302
std        0.620752
min        5.262172
25%        6.395262
50%        6.893656
75%        7.286876
max        8.716044
Name: Price_in_euros, dtype: float64

In [35]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmses = []

for train_idx, valid_idx in kf.split(X_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

    y_tr_log = np.log1p(y_tr)

    model = LGBMRegressor(
        n_estimators=800,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_tr, y_tr_log)

    y_pred_log = model.predict(X_val)
    y_pred = np.expm1(y_pred_log)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    rmses.append(rmse)

print("RMSE promedio:", np.mean(rmses))


RMSE promedio: 0.19763443428611283
